# Dataset testing and statistics
The authoritative automated checks are in `validate_data.py`. This notebook reproduces the headline statistics and visual summaries interactively.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("pelevin_named_entities.csv")

## Core integrity statistics

In [ ]:
summary = {
    "records": len(df),
    "unique_entities": df["entity"].nunique(),
    "works": df["work_title"].nunique(),
    "total_mentions": int(df["mention_count"].sum()),
    "duplicate_work_entity_pairs": int(df.duplicated(["work_title", "entity"]).sum()),
    "character_rows": int(df["exclude_as_character"].sum()),
}
summary

In [ ]:
assert summary == {
    "records": 1456,
    "unique_entities": 745,
    "works": 22,
    "total_mentions": 5673,
    "duplicate_work_entity_pairs": 0,
    "character_rows": 33,
}
print("Core statistics: OK")

## Missing metadata

In [ ]:
df[["activities", "birth_year", "death_year", "country_or_polity"]].isna().sum()

## Records and mentions by work

In [ ]:
by_work = df.groupby(["work_year", "work_title"]).agg(
    records=("entity", "size"),
    mentions=("mention_count", "sum"),
).reset_index().sort_values(["work_year", "work_title"])
by_work

In [ ]:
ax = by_work.plot.bar(x="work_title", y="mentions", figsize=(14, 5), legend=False)
ax.set_xlabel("Work")
ax.set_ylabel("Mentions")
ax.set_title("Named-entity mentions by work")
plt.tight_layout()

## Most common activity labels

In [ ]:
activities = (df.assign(activity=df["activities"].fillna("").str.split("|"))
              .explode("activity")
              .query("activity != ''")
              .drop_duplicates(["entity", "activity"])
              ["activity"].value_counts().head(20))
activities